In [ ]:
import sys, os, random
sys.path.append('/content/genAI-project-24f1002692')

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

from src.dataset.data_loader import load_competition_data, load_rag_resources, load_retriever
from src.models.BiGRU_Rag_scratch import (
    OPTIONS, build_rag_context, build_vocab, MCQDataset, OptionScorer,
)
from src.inference.BiGRU_Rag_inference import (
    train_bigru_rag, predict_top3, save_submission,
)

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
# Load competition data + RAG resources (Wikipedia FAISS index + retriever)
train_dataset, test_dataset, sample_submission = load_competition_data()
wiki_index, wiki_index_df = load_rag_resources()
retriever = load_retriever()

In [ ]:
# Prompt-level train/val split (prevents the same question leaking across both sets)
train_df = train_dataset.copy()
test_df = test_dataset.copy()

prompt_answer = train_df.groupby('prompt')['answer'].first()
unique_prompts = prompt_answer.index.values

train_prompts, val_prompts = train_test_split(
    unique_prompts, test_size=0.15, random_state=SEED, stratify=prompt_answer.values
)

train_split = train_df[train_df['prompt'].isin(train_prompts)].reset_index(drop=True)
val_split   = train_df[train_df['prompt'].isin(val_prompts)].reset_index(drop=True)

overlap = set(train_split['prompt']) & set(val_split['prompt'])
print(f"Train rows: {len(train_split)}  |  Val rows: {len(val_split)}  |  Prompt overlap: {len(overlap)}")

In [ ]:
# Build vocab from train split only, then wrap all 3 splits in MCQDataset + DataLoader
vocab = build_vocab(train_split)
print(f"Vocab size (with RAG context): {len(vocab)}")

train_loader = DataLoader(MCQDataset(train_split, vocab, augment=True), batch_size=32, shuffle=True)
val_loader   = DataLoader(MCQDataset(val_split, vocab, augment=False), batch_size=64, shuffle=False)
train_eval_loader = DataLoader(MCQDataset(train_split, vocab, augment=False), batch_size=64, shuffle=False)
test_loader  = DataLoader(MCQDataset(test_df, vocab, augment=False), batch_size=64, shuffle=False)

In [ ]:
# Build model + train (checkpoints best val-MAP@3 model to best_scratch_rag_model.pt)
model = OptionScorer(vocab_size=len(vocab)).to(device)
print(model)

history, best_map3 = train_bigru_rag(
    model, train_loader, train_eval_loader, val_loader,
    train_split, val_split, device,
    epochs=25, use_wandb=True,
)

In [ ]:
# Load best checkpoint, predict on test.csv, save Kaggle submission
model.load_state_dict(torch.load("best_scratch_rag_model.pt", map_location=device))

test_preds = predict_top3(model, test_loader, device)
submission = save_submission(test_df, test_preds, out_path="submission_scratch_rag.csv")
submission.head(10)